# Study on langchain

Use local ollama service to study langchain

In [6]:
import  os
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print(f'Import LangChain V{langchain.__version__}')

Import LangChain V1.2.7


## Link to Model

In [ ]:
# get api key from local file
with open('api-key/deepseek.txt', 'r') as f:
    api_key = f.read().strip()

In [ ]:
# create LLM client
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-v4-flash",
    temperature=0,
)

In [ ]:
# try out
print(
    StrOutputParser().invoke(
        llm.invoke([("human", "Introduce yourself")])
    )
)

Hey there! 👋 I'm DeepSeek, an AI assistant created by DeepSeek (深度求索). I'm here to help you with all sorts of questions and tasks!

A few things about me:
- **I'm completely free** to use - no hidden fees or subscriptions
- **I can handle huge amounts of text** - my context window is 1 million tokens, so I can process entire books like the Three-Body Problem trilogy in one go
- **I can read files** - upload images, PDFs, Word docs, Excel sheets, PowerPoint presentations, or text files and I'll extract and process the info
- **I can search the web** - if you turn on the web search feature in the app/web interface
- **I'm a text model** - no image recognition, but I can read text from uploaded images
- **My knowledge** goes up to May 2025

I'm conversational, love diving deep into topics, and I'll always be upfront about what I can and can't do. So, what's on your mind? 😊


## Demo from book

In [31]:
prompt_extract  =  ChatPromptTemplate.from_template(
  "请从以下文本中提取技术规格：\n\n{text_input}"
)

prompt_transform  =  ChatPromptTemplate.from_template(
  "请将以下技术规格转为 JSON 格式，包含 'CPU'、'内存' 和 '硬盘' 三个键：\n\n{specifications}"
)

extraction_chain = prompt_extract | llm | StrOutputParser()

full_chain = (
    {"specifications":  extraction_chain}
    | prompt_transform | llm | StrOutputParser()
)

In [32]:
input_text = "新款笔记本配备 3.5GHz 八核处理器、16GB 内存和 1TB NVMe SSD。"

final_result = full_chain.invoke({"text_input":  input_text})

print("\n--- 最终 JSON 输出 ---")
print(final_result)


--- 最终 JSON 输出 ---
```json
{
  "CPU": "3.5GHz 八核",
  "内存": "16GB",
  "硬盘": "1TB NVMe SSD"
}
```


## Try to design another example

* Analysis style and rhetoric from a paragraph
* Summarize into a json

In [35]:
def analysis_chain(model):
    prompt_analysis = ChatPromptTemplate.from_template(
        "Please analysis style and rhetoric from the paragraph: \n\n"
        "{paragraph}"
    )
    prompt_transform = ChatPromptTemplate.from_template(
        "Please transform the analysis result into JSON format, "
        "incluing 'style' and 'rhetoric' keys, using Chinese\n\n"
        "{analysis}"
    )
    analysis = prompt_analysis | model | StrOutputParser()
    return (
        {"analysis": analysis}
        | prompt_transform | model | StrOutputParser()
    )

In [36]:
paragraph = "林野把名字写进风里，把影子交给石砾，把心跳系向那片消失的湖。"
"风蚀的贝壳像白牙，一路啃咬他的鞋底；盐霜像冷笑，一层层揭他的旧疤；黑戈壁像巨兽，一寸寸吞他的水袋。"
"他走，太阳烧他的背；他停，月亮冻他的额；他跪，星群钉他的眼。"
"第七夜，火山口张开黑唇，吐出一圈银白的盐戒。"
"林野伸手，盐粒尖叫着碎成尘；他俯耳，湖魂在泥下低低哭泣。"
"排比的月光：一滴、两滴、千滴，全落在同一道裂缝。"
"排比的喘息：一次、两次、千次，全献给同一口咸水。"
"排比的星：一颗、两颗、千颗，全替他守口如瓶。"
"他把记录仪埋进泥的心脏，让时间做回医生，让盐做回墓碑。"
"回程，沙墙拔地而起，像千万匹脱缰的琥珀马，踏碎他的脚印，嚼烂他的指南针。"
"林野把芯片含在舌下，像含着最后一片冰；他把记忆藏进瞳孔，像藏着最后一颗火；他把名字交给风暴，像交出最后一枚盾。"
"天穹俯身，用星屑缝补他裂开的影子；戈壁侧身，用砾石为他让出一条光的缝隙。"
"他爬，爬得比夜更黑，比盐更涩，比记忆更长。"
"黎明前，他吐出芯片，芯片上只剩一句排比的遗言：我来过，我听见，我带走。"

chain = analysis_chain(llm)

final_result = chain.invoke({"paragraph":  paragraph})

print("\n--- 最终 JSON 输出 ---")
print(final_result)


--- 最终 JSON 输出 ---
{
  "style": [
    "简洁凝练：三句并列，句式整齐，无多余修饰，却蕴含丰富意象。",
    "意象化表达：以“风”“石砾”“消失的湖”等自然物象承载抽象情感，营造出空灵、苍茫的意境。",
    "抒情性：通过“名字”“影子”“心跳”等个人化元素，将主体与自然融为一体，充满孤独、眷恋或决绝的抒情色彩。",
    "神秘与哀愁：“消失的湖”暗示过往的不可追寻，整体氛围带有淡淡的失落与超然。"
  ],
  "rhetoric": [
    "排比：三个“把……给/向……”结构形成排比，增强节奏感与气势，层层递进地强化主体的情感投射。",
    "拟人：“把影子交给石砾”“把心跳系向湖”将无形之物（影子、心跳）当作有生命、可交付的实体，赋予其动作性。",
    "隐喻：包括“把名字写进风里”隐喻遗忘或消散，“把影子交给石砾”隐喻主体融入荒寂的环境，“把心跳系向那片消失的湖”隐喻对逝去之物的执着与牵挂。",
    "象征：整体构成一种精神层面的“献祭”或“归隐”仪式，风、石砾、湖分别象征无常、坚忍与虚无，共同指向一种与自然融合、与过往和解的终极状态。"
  ]
}
